# MONOREPO → Google Drive sync
Run top to bottom. It mounts Drive, then **clones or fast-forward-pulls** every repo into `MyDrive/MONOREPO/`, so Drive holds a real git checkout of each one (not copies that drift).

**One-time setup:** in the left sidebar of Colab open 🔑 *Secrets*, add `GITHUB_TOKEN` (a fine-grained or classic PAT with read access to the repos), and toggle *Notebook access* on. Public repos work without it.

Re-run any time to update. It never force-resets: a repo with local edits in Drive is skipped and reported.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
OWNER   = 'emilyspringerton'
DEST    = '/content/drive/MyDrive/MONOREPO'
# Edit this list. The monorepo root itself is cloned as DEST; each sub-repo goes inside it.
REPOS   = ['EMILY','IDUNA','SHANKPIT','PARENA','WOTAN','DEADWEIGHT','DEADWEIGHT_2','BRAWLPIT',
           'REDGARDEN','OKEMILY','GOLDENBAND','GoblinFoxDragon','WEAKNIGHT_BEDROCK_RACERS','BIG_O']
SHALLOW = False   # True = --depth 1 (much smaller on Drive, no history)

try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = os.environ.get('GITHUB_TOKEN','')

def url(name):
    auth = f'{TOKEN}@' if TOKEN else ''
    return f'https://{auth}github.com/{OWNER}/{name}.git'

def sh(*a, cwd=None):
    r = subprocess.run(a, cwd=cwd, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).replace(TOKEN or '\0', '***').strip()

def sync(name, path):
    if not os.path.isdir(os.path.join(path, '.git')):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        args = ['git','clone'] + (['--depth','1'] if SHALLOW else []) + [url(name), path]
        rc, out = sh(*args)
        return ('cloned' if rc == 0 else 'FAILED'), out.splitlines()[-1] if out else ''
    if sh('git','status','--porcelain', cwd=path)[1]:
        return 'skipped (local edits)', ''
    sh('git','remote','set-url','origin', url(name), cwd=path)
    rc, out = sh('git','pull','--ff-only', cwd=path)
    return ('updated' if rc == 0 else 'FAILED'), out.splitlines()[-1] if out else ''

os.makedirs(DEST, exist_ok=True)
results = [('MONOREPO(root)',) + sync('MONOREPO', DEST)]
for r in REPOS:
    results.append((r,) + sync(r, os.path.join(DEST, r)))
for r in results: print(f'{r[0]:28} {r[1]:24} {r[2]}')

## Using it
- **Colab:** `!ln -sfn /content/drive/MyDrive/MONOREPO /content/MONOREPO` gives you a short path.
- **Your computer:** install *Google Drive for desktop*; `MONOREPO/` appears as a normal folder. Symlink it where you like (`ln -s ~/GoogleDrive/MyDrive/MONOREPO ~/MONOREPO`).
- **Don't run git from two places on the same checkout** (Drive desktop syncing `.git` while Colab pulls can corrupt it). Treat Drive as the read/backup copy; do commits from the VPS/your machine.

In [ ]:
!ln -sfn /content/drive/MyDrive/MONOREPO /content/MONOREPO && ls /content/MONOREPO | head